In [1]:
import os
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.util import ngrams
from nltk.tag.hmm import HiddenMarkovModelTrainer
import pycrfsuite

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


print("=" * 60)
print("TOPIC 1: File Reading, Preprocessing, & Word Embeddings")
print("=" * 60)

t1_file = "nlp_lab_input.txt"
t1_text = (
    "Natural Language Processing allows computers to understand human language.\n"
    "Word embeddings capture rich semantic relationships between words."
)

with open(t1_file, "w", encoding="utf-8") as f:
    f.write(t1_text)

with open(t1_file, "r", encoding="utf-8") as f:
    t1_raw = f.read()

t1_clean = re.sub(r'[^a-z\s]', '', t1_raw.lower())
t1_tokens = word_tokenize(t1_clean)
t1_stopwords = set(stopwords.words('english'))
t1_filtered = [w for w in t1_tokens if w not in t1_stopwords]

print("[Output] Preprocessed Tokens:")
print(t1_filtered)

t1_w2v = Word2Vec([t1_filtered], vector_size=10, window=3, min_count=1, sg=0)
t1_vector = t1_w2v.wv['language']

print("\n[Output] Embedding Vector for 'language' (Dim:", t1_vector.shape[0], "):")
print(t1_vector)

if os.path.exists(t1_file):
    os.remove(t1_file)


print("\n" + "=" * 60)
print("TOPIC 2: TF-IDF with Latent Semantic Indexing (LSI)")
print("=" * 60)

t2_corpus = [
    "Machine learning algorithms learn patterns from dataset records.",
    "Deep learning and natural language processing use neural network architectures.",
    "Data science involves statistical analysis and machine learning techniques."
]

t2_tfidf = TfidfVectorizer(stop_words='english')
t2_matrix = t2_tfidf.fit_transform(t2_corpus)

t2_lsi = TruncatedSVD(n_components=2, random_state=42)
t2_reduced = t2_lsi.fit_transform(t2_matrix)

df_lsi = pd.DataFrame(t2_reduced, columns=["Topic_1", "Topic_2"])
print("[Output] LSI Reduced Matrix:")
print(df_lsi)

t2_features = t2_tfidf.get_feature_names_out()
print("\n[Output] Top terms per topic:")
for idx, topic in enumerate(t2_lsi.components_):
    top_words = [t2_features[i] for i in topic.argsort()[:-4:-1]]
    print(f"  Topic {idx+1}: {', '.join(top_words)}")


print("\n" + "=" * 60)
print("TOPIC 3: N-Grams")
print("=" * 60)

t3_text = "natural language processing creates intelligent software solutions"
t3_words = t3_text.split()

print("[Output] NLTK Bigrams:")
print(list(ngrams(t3_words, 2)))

print("\n[Output] NLTK Trigrams:")
print(list(ngrams(t3_words, 3)))

t3_vectorizer = CountVectorizer(ngram_range=(1, 2))
t3_vectorizer.fit_transform([t3_text])
print("\n[Output] Scikit-Learn Extracted Features (Sample 8):")
print(t3_vectorizer.get_feature_names_out()[:8])


print("\n" + "=" * 60)
print("TOPIC 4: POS Tagging using HMM and CRF")
print("=" * 60)

print("\n--- 4A. HMM POS Tagging ---")

t4_hmm_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")]
]

t4_symbols = sorted(list({pair[0] for sent in t4_hmm_data for pair in sent}))
t4_tags = sorted(list({pair[1] for sent in t4_hmm_data for pair in sent}))

t4_hmm_trainer = HiddenMarkovModelTrainer(states=t4_tags, symbols=t4_symbols)
t4_hmm_tagger = t4_hmm_trainer.train_supervised(t4_hmm_data)

t4_test_sentence = ["The", "dog", "runs"]
print("[Output] HMM Tagging Result:")
print(t4_hmm_tagger.tag(t4_test_sentence))


print("\n--- 4B. CRF POS Tagging ---")

def extract_crf_features(sent, i):
    word = sent[i][0]
    return {
        'word.lower()': word.lower(),
        'is_first': i == 0,
        'is_last': i == len(sent) - 1,
        'is_capitalized': word[0].isupper(),
        'prev_word': '' if i == 0 else sent[i-1][0].lower(),
        'next_word': '' if i == len(sent) - 1 else sent[i+1][0].lower(),
    }

t4_crf_data = [
    [("The", "DET"), ("quick", "ADJ"), ("fox", "NOUN"), ("jumps", "VERB")],
    [("A", "DET"), ("lazy", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")]
]

X_train = [[extract_crf_features(s, i) for i in range(len(s))] for s in t4_crf_data]
y_train = [[tag for word, tag in s] for s in t4_crf_data]

t4_crf_trainer = pycrfsuite.Trainer(verbose=False)
for xseq, yseq in zip(X_train, y_train):
    t4_crf_trainer.append(xseq, yseq)

crf_model_path = 'pos_crf_model.crfsuite'
t4_crf_trainer.set_params({'c1': 1.0, 'c2': 1e-3, 'max_iterations': 50})
t4_crf_trainer.train(crf_model_path)

t4_crf_tagger = pycrfsuite.Tagger()
t4_crf_tagger.open(crf_model_path)

unlabeled_test = [("The", ""), ("lazy", ""), ("fox", "")]
X_test = [extract_crf_features(unlabeled_test, i) for i in range(len(unlabeled_test))]
predictions = t4_crf_tagger.tag(X_test)

print("[Output] CRF Tagging Result:")
print(list(zip([w[0] for w in unlabeled_test], predictions)))

t4_crf_tagger.close()
if os.path.exists(crf_model_path):
    os.remove(crf_model_path)

print("\n" + "=" * 60)
print("Execution Completed Successfully!")
print("=" * 60)

TOPIC 1: File Reading, Preprocessing, & Word Embeddings
[Output] Preprocessed Tokens:
['natural', 'language', 'processing', 'allows', 'computers', 'understand', 'human', 'language', 'word', 'embeddings', 'capture', 'rich', 'semantic', 'relationships', 'words']

[Output] Embedding Vector for 'language' (Dim: 10 ):
[-0.00536309  0.00236467  0.05104123  0.09010639 -0.0930436  -0.07117888
  0.06459852  0.08974349 -0.05016188 -0.03763942]

TOPIC 2: TF-IDF with Latent Semantic Indexing (LSI)
[Output] LSI Reduced Matrix:
    Topic_1   Topic_2
0  0.714320 -0.241331
1  0.396795  0.917714
2  0.709651 -0.270213

[Output] Top terms per topic:
  Topic 1: machine, learning, patterns
  Topic 2: architectures, neural, network

TOPIC 3: N-Grams
[Output] NLTK Bigrams:
[('natural', 'language'), ('language', 'processing'), ('processing', 'creates'), ('creates', 'intelligent'), ('intelligent', 'software'), ('software', 'solutions')]

[Output] NLTK Trigrams:
[('natural', 'language', 'processing'), ('languag

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)


In [5]:
import os
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.util import ngrams
from nltk.tag.hmm import HiddenMarkovModelTrainer
import pycrfsuite

# Download required NLTK assets
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)


# =====================================================================
# TOPIC 1: File Reading, Preprocessing, & Word Embeddings
# =====================================================================
print("=" * 60)
print("TOPIC 1: File Reading, Preprocessing, & Word Embeddings")
print("=" * 60)

t1_file = "nlp_lab_input.txt"
t1_text = (
    "Natural Language Processing allows computers to understand human language.\n"
    "Word embeddings capture rich semantic relationships between words."
)

with open(t1_file, "w", encoding="utf-8") as f:
    f.write(t1_text)

with open(t1_file, "r", encoding="utf-8") as f:
    t1_raw = f.read()

# Preprocessing
t1_clean = re.sub(r'[^a-z\s]', '', t1_raw.lower())
t1_tokens = word_tokenize(t1_clean)
t1_stopwords = set(stopwords.words('english'))
t1_filtered = [w for w in t1_tokens if w not in t1_stopwords]

print("[Output] Preprocessed Tokens:")
print(t1_filtered)

# Word2Vec Embeddings
t1_w2v = Word2Vec([t1_filtered], vector_size=10, window=3, min_count=1, sg=0)
t1_vector = t1_w2v.wv['language']

print("\n[Output] Embedding Vector for 'language' (Dim:", t1_vector.shape[0], "):")
print(t1_vector)

if os.path.exists(t1_file):
    os.remove(t1_file)


# =====================================================================
# TOPIC 2: TF-IDF with Latent Semantic Indexing (LSI)
# =====================================================================
print("\n" + "=" * 60)
print("TOPIC 2: TF-IDF with Latent Semantic Indexing (LSI)")
print("=" * 60)

t2_corpus = [
    "Machine learning algorithms learn patterns from dataset records.",
    "Deep learning and natural language processing use neural network architectures.",
    "Data science involves statistical analysis and machine learning techniques."
]

t2_tfidf = TfidfVectorizer(stop_words='english')
t2_matrix = t2_tfidf.fit_transform(t2_corpus)

# LSI via Truncated SVD
t2_lsi = TruncatedSVD(n_components=2, random_state=42)
t2_reduced = t2_lsi.fit_transform(t2_matrix)

df_lsi = pd.DataFrame(t2_reduced, columns=["Topic_1", "Topic_2"])
print("[Output] LSI Reduced Matrix:")
print(df_lsi)

t2_features = t2_tfidf.get_feature_names_out()
print("\n[Output] Top terms per topic:")
for idx, topic in enumerate(t2_lsi.components_):
    top_words = [t2_features[i] for i in topic.argsort()[:-4:-1]]
    print(f"  Topic {idx+1}: {', '.join(top_words)}")


# =====================================================================
# TOPIC 3: N-Grams
# =====================================================================
print("\n" + "=" * 60)
print("TOPIC 3: N-Grams")
print("=" * 60)

t3_text = "natural language processing creates intelligent software solutions"
t3_words = t3_text.split()

print("[Output] NLTK Bigrams:")
print(list(ngrams(t3_words, 2)))

print("\n[Output] NLTK Trigrams:")
print(list(ngrams(t3_words, 3)))

t3_vectorizer = CountVectorizer(ngram_range=(1, 2))
t3_vectorizer.fit_transform([t3_text])
print("\n[Output] Scikit-Learn Extracted Features (Sample 8):")
print(t3_vectorizer.get_feature_names_out()[:8])


# =====================================================================
# TOPIC 4: POS Tagging using HMM and CRF
# =====================================================================
print("\n" + "=" * 60)
print("TOPIC 4: POS Tagging using HMM and CRF")
print("=" * 60)

# --- 4A. Hidden Markov Model (HMM) ---
print("\n--- 4A. HMM POS Tagging ---")

# Fully isolated training data
t4_hmm_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")]
]

# Safely extract vocabulary and tag sets using indexing [0] and [1]
t4_symbols = sorted(list({pair[0] for sent in t4_hmm_data for pair in sent}))
t4_tags = sorted(list({pair[1] for sent in t4_hmm_data for pair in sent}))

# Train HMM
t4_hmm_trainer = HiddenMarkovModelTrainer(states=t4_tags, symbols=t4_symbols)
t4_hmm_tagger = t4_hmm_trainer.train_supervised(t4_hmm_data)

t4_test_sentence = ["The", "dog", "runs"]
print("[Output] HMM Tagging Result:")
print(t4_hmm_tagger.tag(t4_test_sentence))


# --- 4B. Conditional Random Field (CRF) ---
print("\n--- 4B. CRF POS Tagging ---")

def extract_crf_features(sent, i):
    word = sent[i][0]
    return {
        'word.lower()': word.lower(),
        'is_first': i == 0,
        'is_last': i == len(sent) - 1,
        'is_capitalized': word[0].isupper(),
        'prev_word': '' if i == 0 else sent[i-1][0].lower(),
        'next_word': '' if i == len(sent) - 1 else sent[i+1][0].lower(),
    }

t4_crf_data = [
    [("The", "DET"), ("quick", "ADJ"), ("fox", "NOUN"), ("jumps", "VERB")],
    [("A", "DET"), ("lazy", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")]
]

X_train = [[extract_crf_features(s, i) for i in range(len(s))] for s in t4_crf_data]
y_train = [[tag for word, tag in s] for s in t4_crf_data]

t4_crf_trainer = pycrfsuite.Trainer(verbose=False)
for xseq, yseq in zip(X_train, y_train):
    t4_crf_trainer.append(xseq, yseq)

crf_model_path = 'pos_crf_model.crfsuite'
t4_crf_trainer.set_params({'c1': 1.0, 'c2': 1e-3, 'max_iterations': 50})
t4_crf_trainer.train(crf_model_path)

# Inference
t4_crf_tagger = pycrfsuite.Tagger()
t4_crf_tagger.open(crf_model_path)

unlabeled_test = [("The", ""), ("lazy", ""), ("fox", "")]
X_test = [extract_crf_features(unlabeled_test, i) for i in range(len(unlabeled_test))]
predictions = t4_crf_tagger.tag(X_test)

print("[Output] CRF Tagging Result:")
print(list(zip([w[0] for w in unlabeled_test], predictions)))

t4_crf_tagger.close()
if os.path.exists(crf_model_path):
    os.remove(crf_model_path)

print("\n" + "=" * 60)
print("Execution Completed Successfully!")
print("=" * 60)

TOPIC 1: File Reading, Preprocessing, & Word Embeddings
[Output] Preprocessed Tokens:
['natural', 'language', 'processing', 'allows', 'computers', 'understand', 'human', 'language', 'word', 'embeddings', 'capture', 'rich', 'semantic', 'relationships', 'words']

[Output] Embedding Vector for 'language' (Dim: 10 ):
[-0.00536309  0.00236467  0.05104123  0.09010639 -0.0930436  -0.07117888
  0.06459852  0.08974349 -0.05016188 -0.03763942]

TOPIC 2: TF-IDF with Latent Semantic Indexing (LSI)
[Output] LSI Reduced Matrix:
    Topic_1   Topic_2
0  0.714320 -0.241331
1  0.396795  0.917714
2  0.709651 -0.270213

[Output] Top terms per topic:
  Topic 1: machine, learning, patterns
  Topic 2: architectures, neural, network

TOPIC 3: N-Grams
[Output] NLTK Bigrams:
[('natural', 'language'), ('language', 'processing'), ('processing', 'creates'), ('creates', 'intelligent'), ('intelligent', 'software'), ('software', 'solutions')]

[Output] NLTK Trigrams:
[('natural', 'language', 'processing'), ('languag

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)


In [28]:
import os
import pandas as pd
import re
import gensim
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

corpus = [
    "Natural Language Processing allows computers to understand human language.\n",
    "Word embeddings capture rich semantic relationships between words."
]

with open("nlpsamplelab2", "w", encoding="utf-8") as f:
    f.writelines(corpus)
with open("nlpsamplelab2", "r", encoding="utf-8") as f:
    data = f.readlines()
print("From text file: ")
print(data)

print("Preprocessing")
clean1data= [re.sub(r"^[a-z]\s", "", sen ) for sen in data]
lowerdata = [sen.lower() for sen in clean1data]
tokens = word_tokenize(" ".join(lowerdata))
stop_words = set(stopwords.words('english'))
clean2token = [w for w in tokens if not w in stop_words]
print(clean2token)

converter = gensim.models.Word2Vec([clean2token], vector_size=10, window=3, min_count=1, sg=0)
print("Word embedding for 'computers':")
print(converter.wv['computers'])

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(lowerdata)
print("TF-IDF Matrix:")
print(pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out()))

lsi = TruncatedSVD(n_components=2, random_state=42)
lsi_matrix = lsi.fit_transform(tfidf_matrix)
df_lsi = pd.DataFrame(lsi_matrix, columns=["Topic_1", "Topic_2"])
print("LSI Reduced Matrix:")
print(df_lsi)
print("Feature names: ")
print(tfidf_vectorizer.get_feature_names_out())


From text file: 
['Natural Language Processing allows computers to understand human language.\n', 'Word embeddings capture rich semantic relationships between words.']
Preprocessing
['natural', 'language', 'processing', 'allows', 'computers', 'understand', 'human', 'language', '.', 'word', 'embeddings', 'capture', 'rich', 'semantic', 'relationships', 'words', '.']
Word embedding for 'computers':
[-0.00410223 -0.08368949 -0.05600012  0.07104538  0.0335254   0.0722567
  0.06800248  0.07530741 -0.03789154 -0.00561806]
TF-IDF Matrix:
     allows   capture  computers  ...  understand      word     words
0  0.316228  0.000000   0.316228  ...    0.316228  0.000000  0.000000
1  0.000000  0.377964   0.000000  ...    0.000000  0.377964  0.377964

[2 rows x 14 columns]
LSI Reduced Matrix:
        Topic_1       Topic_2
0 -6.957682e-17  1.000000e+00
1  1.000000e+00 -4.342464e-16
Feature names: 
['allows' 'capture' 'computers' 'embeddings' 'human' 'language' 'natural'
 'processing' 'relationships' '

In [1]:
import numpy as np
import nltk
from collections import Counter

# NLTK Imports
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, treebank
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.util import ngrams
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from nltk.tag import hmm

# Scikit-Learn Imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


def main():
    # ---------------------------------------------------------
    # 0. Download NLTK Resources
    # ---------------------------------------------------------
    print("Downloading necessary NLTK datasets...")
    resources = ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'treebank']
    for resource in resources:
        nltk.download(resource, quiet=True)
    print("Downloads complete.\n" + "=" * 50)

    # ---------------------------------------------------------
    # 1. Split Sentences Based on Phases of NLP
    # ---------------------------------------------------------
    print("\n--- TASK 1: PHASES OF NLP ---")
    text = "Natural Language Processing is a fascinating field. It enables computers to process human language efficiently."

    # Phase 1: Sentence Segmentation
    sentences = sent_tokenize(text)
    print("\n1. Sentence Segmentation:")
    for i, sent in enumerate(sentences, 1):
        print(f"  Sentence {i}: {sent}")

    # Phase 2: Word Tokenization
    tokens = [word_tokenize(sent) for sent in sentences]
    print("\n2. Word Tokenization:")
    print(" ", tokens)

    # Phase 3: Lowercasing & Stopword Removal
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [
        [word.lower() for word in sent if word.isalnum() and word.lower() not in stop_words]
        for sent in tokens
    ]
    print("\n3. Stopword Removal & Lowercasing:")
    print(" ", filtered_tokens)

    # Phase 4: Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [
        [lemmatizer.lemmatize(word) for word in sent]
        for sent in filtered_tokens
    ]
    print("\n4. Lemmatization:")
    print(" ", lemmatized_tokens)

    # Phase 5: POS Tagging
    pos_tags = [pos_tag(sent) for sent in tokens]
    print("\n5. Part-of-Speech (POS) Tagging:")
    for i, tags in enumerate(pos_tags, 1):
        print(f"  Sentence {i} POS: {tags}")

    print("\n" + "=" * 50)

    # ---------------------------------------------------------
    # 2. Check Occurrences of 1-gram, 2-gram and Calculate Measures
    # ---------------------------------------------------------
    print("\n--- TASK 2: N-GRAMS AND MEASURES ---")
    
    # Flatten sentences into a single clean list of words
    corpus_words = [word.lower() for sent in tokens for word in sent if word.isalnum()]

    # 1-gram (Unigrams)
    unigrams = list(ngrams(corpus_words, 1))
    unigram_freq = Counter(unigrams)
    print("\n1-Gram Frequencies:")
    for ug, count in unigram_freq.items():
        print(f"  {ug[0]}: {count}")

    # 2-gram (Bigrams)
    bigrams = list(ngrams(corpus_words, 2))
    bigram_freq = Counter(bigrams)
    print("\n2-Gram Frequencies:")
    for bg, count in bigram_freq.items():
        print(f"  {bg}: {count}")

    # Collocation Measures (Pointwise Mutual Information - PMI)
    finder = BigramCollocationFinder.from_words(corpus_words)
    pmi_scores = finder.score_ngrams(BigramAssocMeasures.pmi)
    
    print("\nBigram Association Measures (PMI Scores):")
    for bigram, score in pmi_scores:
        print(f"  Bigram: {bigram} | PMI Score: {score:.4f}")

    print("\n" + "=" * 50)

    # ---------------------------------------------------------
    # 3. Test Sentence Evaluation: HMM & LSI using SVD Correlation
    # ---------------------------------------------------------
    print("\n--- TASK 3: HMM TAGGING & LSI SVD CORRELATION ---")

    # --- Part A: HMM Tagging ---
    print("\n[A] HMM POS Tagging")
    train_data = treebank.tagged_sents()[:500]  # Fast subset training
    hmm_trainer = hmm.HiddenMarkovModelTrainer()
    hmm_tagger = hmm_trainer.train_supervised(train_data)

    test_sentence = "Computers analyze human language"
    test_tokens = word_tokenize(test_sentence)

    hmm_predictions = hmm_tagger.tag(test_tokens)
    print(f"Test Sentence: '{test_sentence}'")
    print("HMM Tag Predictions:", hmm_predictions)

    # --- Part B: LSI with SVD & Pearson Correlation ---
    print("\n[B] LSI via SVD & Pearson Correlation")
    corpus_docs = [
        "Natural Language Processing is a fascinating field.",
        "Computers process human language efficiently using AI.",
        "Machine learning algorithms classify text and sequence data."
    ]

    # 1. TF-IDF Representation
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus_docs)

    # 2. Singular Value Decomposition (SVD) for LSI
    svd = TruncatedSVD(n_components=2, random_state=42)
    corpus_lsi = svd.fit_transform(tfidf_matrix)

    # 3. Project test sentence to LSI latent space
    test_tfidf = vectorizer.transform([test_sentence])
    test_lsi = svd.transform(test_tfidf)

    print(f"\nTest Sentence Latent Vector (2D SVD Space):\n  {test_lsi[0]}")

    # 4. Compute Pearson Correlation
    print("\nCorrelation of Test Sentence with Corpus Documents:")
    for i, doc_vec in enumerate(corpus_lsi):
        corr_matrix = np.corrcoef(test_lsi[0], doc_vec)
        correlation = corr_matrix[0, 1]
        print(f"  Doc {i+1}: '{corpus_docs[i]}'")
        print(f"  --> Pearson Correlation: {correlation:.4f}\n")


if __name__ == "__main__":
    main()

Downloads complete.

--- TASK 1: PHASES OF NLP ---

1. Sentence Segmentation:
  Sentence 1: Natural Language Processing is a fascinating field.
  Sentence 2: It enables computers to process human language efficiently.

2. Word Tokenization:
  [['Natural', 'Language', 'Processing', 'is', 'a', 'fascinating', 'field', '.'], ['It', 'enables', 'computers', 'to', 'process', 'human', 'language', 'efficiently', '.']]

3. Stopword Removal & Lowercasing:
  [['natural', 'language', 'processing', 'fascinating', 'field'], ['enables', 'computers', 'process', 'human', 'language', 'efficiently']]

4. Lemmatization:
  [['natural', 'language', 'processing', 'fascinating', 'field'], ['enables', 'computer', 'process', 'human', 'language', 'efficiently']]

5. Part-of-Speech (POS) Tagging:
  Sentence 1 POS: [('Natural', 'JJ'), ('Language', 'NNP'), ('Processing', 'NNP'), ('is', 'VBZ'), ('a', 'DT'), ('fascinating', 'JJ'), ('field', 'NN'), ('.', '.')]
  Sentence 2 POS: [('It', 'PRP'), ('enables', 'VBZ'), ('com

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)


Test Sentence: 'Computers analyze human language'
HMM Tag Predictions: [('Computers', 'NNP'), ('analyze', 'NNP'), ('human', 'NNP'), ('language', 'NNP')]

[B] LSI via SVD & Pearson Correlation

Test Sentence Latent Vector (2D SVD Space):
  [ 5.25992514e-01 -6.12210542e-16]

Correlation of Test Sentence with Corpus Documents:
  Doc 1: 'Natural Language Processing is a fascinating field.'
  --> Pearson Correlation: 1.0000

  Doc 2: 'Computers process human language efficiently using AI.'
  --> Pearson Correlation: 1.0000

  Doc 3: 'Machine learning algorithms classify text and sequence data.'
  --> Pearson Correlation: -1.0000



C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\nltk\tag\hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


In [ ]:
print("--- TASK 3: HMM TAGGING & LSI SVD CORRELATION (WITHOUT TREEBANK) ---")

print("\n[A] HMM POS Tagging")

train_data = [
    [('Natural', 'ADJ'), ('Language', 'NOUN'), ('Processing', 'NOUN'), ('is', 'VERB'), ('a', 'DET'), ('fascinating', 'ADJ'), ('field', 'NOUN')],
    [('Computers', 'NOUN'), ('process', 'VERB'), ('human', 'ADJ'), ('language', 'NOUN'), ('efficiently', 'ADV')],
    [('Machine', 'NOUN'), ('learning', 'NOUN'), ('algorithms', 'NOUN'), ('analyze', 'VERB'), ('text', 'NOUN')],
    [('Computers', 'NOUN'), ('analyze', 'VERB'), ('human', 'ADJ'), ('language', 'NOUN')],
    [('AI', 'NOUN'), ('models', 'NOUN'), ('predict', 'VERB'), ('words', 'NOUN')]
]

# Train HMM Tagger on custom supervised data
hmm_trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = hmm_trainer.train_supervised(train_data)

# Test sentence processing
test_sentence = "Computers analyze human language"
test_tokens = word_tokenize(test_sentence)

# Perform HMM Tagging
hmm_predictions = hmm_tagger.tag(test_tokens)
print(f"Test Sentence: '{test_sentence}'")
print("HMM Tag Predictions:", hmm_predictions)


# =========================================================
# PART B: LSI using SVD & Pearson Correlation
# =========================================================
print("\n[B] LSI via SVD & Pearson Correlation")

# Reference Corpus Documents
corpus_docs = [
    "Natural Language Processing is a fascinating field.",
    "Computers process human language efficiently using AI.",
    "Machine learning algorithms classify text and sequence data."
]

# 1. Generate TF-IDF Matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_docs)

# 2. Perform Singular Value Decomposition (SVD) for LSI
n_components = 2
svd = TruncatedSVD(n_components=n_components, random_state=42)
corpus_lsi = svd.fit_transform(tfidf_matrix)

# 3. Transform Test Sentence into 2D Latent Vector Space
test_tfidf = vectorizer.transform([test_sentence])
test_lsi = svd.transform(test_tfidf)

print(f"\nTest Sentence Latent Vector (2D SVD Space):\n  {test_lsi[0]}")

# 4. Compute Pearson Correlation with each Document
print("\nCorrelation of Test Sentence with Corpus Documents:")
for i, doc_vec in enumerate(corpus_lsi):
    # Calculate Pearson Correlation Matrix between vectors
    corr_matrix = np.corrcoef(test_lsi[0], doc_vec)
    correlation = corr_matrix[0, 1]
    
    print(f"  Doc {i+1}: '{corpus_docs[i]}'")
    print(f"  --> Pearson Correlation: {correlation:.4f}\n")

--- TASK 3: HMM TAGGING & LSI SVD CORRELATION (WITHOUT TREEBANK) ---

[A] HMM POS Tagging
Test Sentence: 'Computers analyze human language'
HMM Tag Predictions: [('Computers', 'NOUN'), ('analyze', 'VERB'), ('human', 'ADJ'), ('language', 'NOUN')]

[B] LSI via SVD & Pearson Correlation

Test Sentence Latent Vector (2D SVD Space):
  [ 5.25992514e-01 -6.12210542e-16]

Correlation of Test Sentence with Corpus Documents:
  Doc 1: 'Natural Language Processing is a fascinating field.'
  --> Pearson Correlation: 1.0000

  Doc 2: 'Computers process human language efficiently using AI.'
  --> Pearson Correlation: 1.0000

  Doc 3: 'Machine learning algorithms classify text and sequence data.'
  --> Pearson Correlation: -1.0000

